# Analysis of Emergency Obstetric Care (EmOC) in Pereira, Colombia
> Note: This notebook requires the [environment dependencies](requirements.txt) to be installed
> as well as either an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local instance of the ORS server.

## Model Summary:

This notebook provides the means to generate a dataset that is described in the [model documentation](../pereira/dataset-interpretability.md).

## Workflow Summary:

The notebook gives an overview of the distribution of centres offering EmOC in the city, their classification and how they can be accessed during an emergency. Open source data from OpenStreetMap and tools (such as the openrouteservice) were used to create accessibility measures. Spatial analysis and other data analytics functions led to generating outputs within the 100x100m grid cells that categorised them into three levels: low, medium, and high.

* **Preprocessing**: Get data for EmOC facilities.
* **Analysis for Offer**:
    * Filter or classify EmOC facilities based on discussed criteria.
    * Visualise EmOC faccilities in their categories.
* **Analysis for Accessibility**:
    * Compute travel times to facilities using openrouteservice API or other routing services.
    * Generate areas for low, medium and high categories based on discussed criteria.
* **Analysis for Demmand**:
    * Downscale the popluation data to the 100x100m grid cells.
    * Derive socio-economic descriptors based on discussed criteria.

* **Result**: Generate results as GIS-compatible files.


### Datasets and Tools:
* [openrouteservice](https://openrouteservice.org/) - generate isochrones on the OpenStreetMap road network

#  Workflow

Make sure you have the required packages installed. You can install them using pip:

```bash
pip install -r path/to/requirements.txt
```

This study integrates various Python geospatial analysis libraries and packages to support spatial data processing, visualization, and isochrone generation. The os module is used to interact with the operating system, managing file paths and reading environment variables such as API keys. folium library along with its MarkerCluster plugin, facilitates the creation of interactive maps for visualizing large-scale geospatial data. The openrouteservice.client serves as an interface to the OpenRouteService API, enabling the extraction of isochrones. pandas library for data analysis, provides functions for analyzing, cleaning, exploring, and manipulating data, while fiona supports reading and writing real-world data using multi-layered GIS formats, such as shapefiles. The shapely package is employed for the manipulation and analysis of planar geometric objects.

## Setting up the virtual environment

```bash
# Create a new virtual environment
python -m venv .venv
activate .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [4]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point

from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler

### Setting up the public API Key from OpenRouteService
In this study, users must obtain an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.


### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [2]:
# Read the api key from the .env file
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

cannot find .env file


ValueError: No API key was specified. Please visit https://openrouteservice.org/sign-up to create one.

### Setting up relevant processing folders

There are different data sources used across the notebook. To handle these data sets, it is recommended to use three directories for input, temp and output data. Some of the files are related to healthcare facilities, population data. The healthcare facilities data is usualy the result of gathering global or national datasets and then carrying out local validation according to the local context. 

Despite being official, administrative boundaries may not reflect the actual patterns of human settlement or economic activity. Therefore, the team used the Functional Urban Area (FUA) as a complementary definition of the study areas. The FUA is defined by [the Joint Research Centre of the European Commission](https://commission.europa.eu/about/departments-and-executive-agencies/joint-research-centre_en) as the actual urban sprawl and human activities, encompassing the core city and economically or socially integrated surrounding regions. The FUA was obtained from [the Global Human Settlement Layer (GHSL) ](https://human-settlement.emergency.copernicus.eu/)dataset, which provides spatial data for functional urban areas worldwide. 

The following datasets are considered as input data for the analysis:


* [Datasets of health facilities](../scripts/Kano/data-inputs/healthcare_facilities.geojson)
* [Population: Women in childbearing age](../scripts/Kano/data-inputs/kano_nga_f_15_49_2015_1km.tif) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447)
* [Study Area](../../../docs/study-areas/grid-boundary-kano.gpkg) defined by the IDEAMAPS team

In [5]:
# Set paths to access Kano data
# Define directories
data_inputs = '../scripts/Pereira/data-inputs/'
data_temp = '../scripts/Pereira/data-temp/'
model_outputs = '../Pereira/'

## 1. Data Collection

### Validated healthcare facilities - (Supply/Offer)
For Kano, the classification for validation was determined based on the project's researchers and their local context, based on data obtained from the [datasets of health facilities](https://www.datos.gov.co/Salud-y-Protecci-n-Social/Registro-Especial-de-Prestadores-y-Sedes-de-Servic/c36g-9fc2/about_data).

In [9]:
healthcare_facilities_validated = gpd.read_file(data_inputs + 'healthcare_facilities.geojson')


In [10]:
healthcare_facilities_validated = healthcare_facilities_validated[healthcare_facilities_validated['project_validation'] != 'No EmOC']
healthcare_facilities_validated

,CodigoPrestador,primary_hc,basic_emoc,comp_emoc,NombrePrestador,CodigoHabilitacionSede,NombreSede,TipoIdentificacion,NumeroIdentificacion,NaturalezaJuridica,...,EmailSede,TelefonoSede,ClasePrestadorDesc,FechaCorte,address_geocoding,status,latitude,longitude,project_validation,geometry
222,6600100217,None,x,?,CAJA DE COMPENSACION FAMILIAR DE RISARALDA COM...,6.600100e+11,CLINICA COMFAMILIAR,NI,891480000,Privada,...,comfarda@comfamiliar.com,3138700 3135600 EXT 2324,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"AV CIRCUNVALAR # 3-01, PEREIRA, Risaralda, COL...",OK,4.807561,-75.680904,Private Comprehensive EmOC,POINT (-75.6809 4.80756)
255,6600100299,None,x,x,CRUZ ROJA SECCIONAL RISARALDA,6.600100e+11,CRUZ ROJA SECCIONAL RISARALDA,NI,891408031,Privada,...,saludrisaralda@cruzrojacolombiana.org - risara...,3498730,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"CLL 16 CRA 15 ESQUINA, PEREIRA, Risaralda, COL...",OK,4.807564,-75.692451,Private Comprehensive EmOC,POINT (-75.69245 4.80756)
271,6600100332,None,x,x,EMPRESA SOCIAL DEL ESTADO SALUD PEREIRA,6.600100e+11,HOSPITAL DE KENNEDY,NI,816005003,P√∫blica,...,calidad@esepereira.gov.co,3515252 ext 501-520,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"CRA 12 CALLE 9 PLAZUELA ESTADIO MORA MORA, PER...",OK,4.815743,-75.733243,Public Comprehensive EmOC,POINT (-75.73324 4.81574)
272,6600100332,None,x,x,EMPRESA SOCIAL DEL ESTADO SALUD PEREIRA,6.600100e+11,HOSPITAL SAN JOAQUIN,NI,816005003,P√∫blica,...,calidad@esepereira.gov.co,3515252 ext 501-520,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,Carrera 26 # 77 - 32 Sector SAN JOAQUIN HOSPIT...,OK,4.797746,-75.742104,Public Comprehensive EmOC,POINT (-75.7421 4.79775)
309,6600100361,None,x,None,CLINICA LOS ROSALES S.A,6.600100e+11,CLINICA LOS ROSALES S.A,NI,891409981,Privada,...,gerencia@clirosales.com,None,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"CRA 9 # 25-25, PEREIRA, Risaralda, COLOMBIA",OK,4.813463,-75.699744,Private Basic EmOC,POINT (-75.69974 4.81346)
424,6600100762,None,None,x,EMPRESA SOCIAL DEL ESTADO HOSPITAL UNIVERSITAR...,6.600100e+11,EMPRESA SOCIAL DEL ESTADO HOSPITAL UNIVERSITAR...,NI,800231235,P√∫blica,...,gerencia@husj.gov.co,3206745,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"Kr 4 No. 24-88, PEREIRA, Risaralda, COLOMBIA",OK,4.818055,-75.698894,Public Comprehensive EmOC,POINT (-75.69889 4.81805)
544,6600101308,None,x,None,REHABILITACION MEDICA INTEGRAL DEL EJE CAFETER...,6.600100e+11,CLINICA COMFAMILIAR,NI,900096895,Privada,...,rmiltda@yahoo.es,3313751,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"AV. CIRCUNVALAR # 3-01, PEREIRA, Risaralda, CO...",OK,4.807561,-75.680904,Private Basic EmOC,POINT (-75.6809 4.80756)
545,6600101308,None,x,None,REHABILITACION MEDICA INTEGRAL DEL EJE CAFETER...,6.600100e+11,RMI S.A.S.,NI,900096895,Privada,...,rmicentro@gmail.com rmiltda@yahoo.es,3164078,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,CARREA 5 N¬∞ 18-33 LOBBY LOCAL 7 CENTRO DE ESP...,OK,4.816227,-75.693636,Private Basic EmOC,POINT (-75.69364 4.81623)
649,6600101587,None,x,None,Sociedad Comercializadora De Insumos y Servici...,6.600100e+11,Clinica San Rafael sede CUBA,NI,900342064,Privada,...,lideraseguramientocalidad@socimedicos.com,3270700 3115411,Instituciones Prestadoras de Servicios de Salu...,Fecha corte REPS: Jul 4 2023 1:19PM,"Carrera 25 #74 A 87, PEREIRA, Risaralda, COLOMBIA",OK,4.801610,-75.741407,Private Basic EmOC,POINT (-75.74141 4.80161)
650,6600101587,None,x,None,Sociedad Comercializadora De Insumos y Servici...,6.600100e+11,Clinica San Rafael sede Megacentro,NI,900342064,Privada,...,lideraseguramientocalidad@socimedicos.com,3270700 3115411,Instituciones Prestadoras de Servicios de Salu...,Fecha corte

In [21]:
# Filtered out facilities that do not provide EmOC services
# to a new geo_json file
healthcare_facilities_validated.to_file(data_temp + 'healthcare_facilities_emoc.geojson', driver='GeoJSON')


### Population Grid Data (Demand)
This data originally comes as a grid (1km resolution) from [WorldPop](https://hub.worldpop.org/geodata/summary?id=18447) to transform it into a 100x100m grid, we use a procedure explained below. 

Note: explain the process to scale down the population data. 
note: explain the rational for female population between 15-49 years old.

In [11]:
study_area = gpd.read_file(data_inputs + '100mGrid.gpkg')
raster_path = data_inputs + 'col_f_15_49_2015_1km.tif'

Clipping the population data to our study area

In [12]:
with rasterio.open(raster_path) as dataset:
    geometries = [study_area.geometry.union_all().__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

In [13]:
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

In [14]:
with rasterio.open(data_inputs + 'pereira_col_f_15_49_2015_1km.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

Calculating the centroids for grid cells

In [15]:
rows, cols = np.where(band1 > 0)
grid_cells = [clipped_transform * (col + 0.5, row + 0.5) for row, col in zip(rows, cols)]
population_values = band1[rows, cols]

In [16]:
grid_df = pd.DataFrame(grid_cells, columns=["longitude", "latitude"])
grid_df["population"] = population_values

grid_df["rowid"] = range(1, len(grid_df) + 1)
population_centroids_gdf = gpd.GeoDataFrame(grid_df, geometry=[Point(xy) for xy in zip(grid_df["longitude"], grid_df["latitude"])])
population_centroids_gdf.set_crs("EPSG:4326", inplace=True)

population_centroids_gdf.to_file(data_temp + "population_centroids.gpkg", driver="GPKG")

In [17]:
population_centroids_gdf

,longitude,latitude,population,rowid,geometry
0,-75.637916,4.926250,42.862034,1,POINT (-75.63792 4.92625)
1,-75.629583,4.926250,51.745846,2,POINT (-75.62958 4.92625)
2,-75.621250,4.926250,45.137421,3,POINT (-75.62125 4.92625)
3,-75.637916,4.917917,55.760937,4,POINT (-75.63792 4.91792)
4,-75.629583,4.917917,83.426140,5,POINT (-75.62958 4.91792)
...,...,...,...,...,...
300,-75.729583,4.726250,91.942818,301,POINT (-75.72958 4.72625)
301,-75.721250,4.726250,145.453781,302,POINT (-75.72125 4.72625)
302,-75.712916,4.726250,162.360748,303,POINT (-75.71292 4.72625)
303,-75.704583,4.726250,101.250107,304,POINT (-75.70458 4.72625)


### Adding population data at 1km grid to 100m grid

In [18]:
# reading in geotiff file as numpy array
def read_tif(file: Path):
    if not file.exists():
        raise FileNotFoundError(f'File {file} not found')

    with rasterio.open(file) as dataset:
        arr = dataset.read()  # (bands X height X width)
        nodata = dataset.nodata
        transform = dataset.transform
        crs = dataset.crs

    # Replace NoData value with NaN
    if nodata is not None:
        arr[arr == nodata] = np.nan

    return arr.transpose((1, 2, 0)), transform, crs

def raster2vector(arr, transform, crs) -> gpd.GeoDataFrame:
    height, width, bands = arr.shape

    # Generate pixel coordinates
    geometries = []
    pixel_values = []

    for row in range(height):
        for col in range(width):
            x_min, y_max = transform * (col, row)  # Top-left corner
            x_max, y_min = transform * (col + 1, row + 1)  # Bottom-right corner

            pixel_value = arr[row, col].tolist()[0]  # Convert numpy array to list
            polygon = Polygon([(x_min, y_max), (x_max, y_max), (x_max, y_min), (x_min, y_min)])

            geometries.append(polygon)
            pixel_values.append(pixel_value)

    # Convert to DataFrame
    gdf = gpd.GeoDataFrame({'pop_grid_pop': pixel_values, 'geometry': geometries}, crs=crs)

    return gdf

# Use the respective EPSG code for Colombia UTM Zone 18N
epsg = 'EPSG:21818'

In [19]:
# Preparing grid
grid_file = data_inputs + '100mGrid.gpkg'
grid = gpd.read_file(grid_file)
grid = grid.to_crs(epsg)
grid['grid_id'] = range(len(grid))
grid = grid[['grid_id', 'rowid', 'geometry', 'latitude', 'lat_min', 'lat_max', 'longitude', 'lon_min','lon_max']].set_geometry('geometry')
grid

,grid_id,rowid,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,1,"POLYGON ((428288.012 544939.864, 428281.792 54...",4.927320,4.926915,4.927725,-75.643963,-75.644491,-75.643434
1,1,2,"POLYGON ((428398.892 544939.757, 428392.673 54...",4.927320,4.926915,4.927725,-75.642963,-75.643491,-75.642434
2,2,3,"POLYGON ((428509.772 544939.65, 428503.553 545...",4.927320,4.926915,4.927725,-75.641963,-75.642491,-75.641434
3,3,4,"POLYGON ((428620.652 544939.543, 428614.433 54...",4.927320,4.926915,4.927725,-75.640962,-75.641491,-75.640434
4,4,5,"POLYGON ((428731.532 544939.437, 428725.313 54...",4.927320,4.926915,4.927725,-75.639962,-75.640491,-75.639434
...,...,...,...,...,...,...,...,...,...
25395,25395,25396,"POLYGON ((422596.86 522653.1, 422590.898 52274...",4.725662,4.725257,4.726067,-75.695086,-75.695613,-75.694559
25396,25396,25397,"POLYGON ((422707.754 522652.989, 422701.791 52...",4.725662,4.725257,4.726067,-75.694086,-75.694613,-75.693559
25397,25397,25398,"POLYGON ((422818.647 522652.879, 422812.685 52...",4.725662,4.725257,4.726067,-75.693086,-75.693613,-75.692559
25398,25398,25399,"POLYGON ((422929.541 522652.768, 422923.579 52...",4.725662,4.725257,4.726067,-75.692086,-75.692614,-75.691559


Building footprint data is used to estimate population distribution within each 1km cell. We recommend using open-source building footprint data from the [Overture Map Foundation](https://overturemaps.org/). Building centroids are spatially joined to a 100 m resolution grid, and the number of buildings within each 100 m cell (bcount) is subsequently calculated.


Downoad the building footprints data from Overture Maps. It uses the bounding box of the study area to limit the download siz (See the bounding box coordinates below). The dataset is natively provided as geoparket. We use the theme = buildings to filter only building footprints. This action is carried out using the Overturemaps python CLI. Therefore, run the following command in your terminal:

```bash
overturemaps download --bbox=-75.850416,4.725257,-75.590559,4.930417 -f geojson --type=building -o ../scripts/Pereira/data-temp/Pereira_GOBv3.geojson
```
**Note**: Make sure your terminal is pointed to the correct path where the notebook is located before running the command above. You might need to navigate as follows 
```bash
cd models/emergency-maternal-care/scripts/
overturemaps download --bbox=-75.850416,4.725257,-75.590559,4.930417 -f geojson --type=building -o ../scripts/Pereira/data-temp/Pereira_GOBv3.geojson
```

In [20]:
# Count buildings per grid cell

# Loading Google building footprints
building_file = data_temp + 'Pereira_GOBv3.geojson'
buildings = gpd.read_file(building_file)
buildings = buildings.to_crs(epsg)
buildings['centroid'] = buildings['geometry'].centroid

# Joining buildings to grid
grid_buildings = grid.sjoin(buildings.set_geometry('centroid').drop(columns='geometry'), how='inner', predicate='intersects')
grid_buildings = grid_buildings.groupby('grid_id')

# Counting buildings per grid
building_counts = grid_buildings.size().rename('bcount')

# Adding building count to grid cells
grid = grid.merge(building_counts, on='grid_id', how='left')

# Assign building count 0 to cells with no buildings (NaN)
grid['bcount'] = grid['bcount'].fillna(0)
grid

,grid_id,rowid,geometry,latitude,lat_min,lat_max,longitude,lon_min,lon_max,bcount
0,0,1,"POLYGON ((428288.012 544939.864, 428281.792 54...",4.927320,4.926915,4.927725,-75.643963,-75.644491,-75.643434,0.0
1,1,2,"POLYGON ((428398.892 544939.757, 428392.673 54...",4.927320,4.926915,4.927725,-75.642963,-75.643491,-75.642434,0.0
2,2,3,"POLYGON ((428509.772 544939.65, 428503.553 545...",4.927320,4.926915,4.927725,-75.641963,-75.642491,-75.641434,0.0
3,3,4,"POLYGON ((428620.652 544939.543, 428614.433 54...",4.927320,4.926915,4.927725,-75.640962,-75.641491,-75.640434,0.0
4,4,5,"POLYGON ((428731.532 544939.437, 428725.313 54...",4.927320,4.926915,4.927725,-75.639962,-75.640491,-75.639434,0.0
...,...,...,...,...,...,...,...,...,...,...
25395,25395,25396,"POLYGON ((422596.86 522653.1, 422590.898 52274...",4.725662,4.725257,4.726067,-75.695086,-75.695613,-75.694559,3.0
25396,25396,25397,"POLYGON ((422707.754 522652.989, 422701.791 52...",4.725662,4.725257,4.726067,-75.694086,-75.694613,-75.693559,3.0
25397,25397,25398,"POLYGON ((422818.647 522652.879, 422812.685 52...",4.725662,4.725257,4.726067,-75.693086,-75.693613,-75.692559,1.0
25398,25398,25399,"POLYGON ((422929.541 522652.768, 422923.579 52...",4.725662,4.725257,4.726067,-75.692086,-75.692614,-75.691559,4.0


The population of each 1km grid is distributed to underlying 100m cells proportionally based on building density. Each 100m grid is assigned a weight equal to its share of the total building count within the 1km grid.

In [21]:
# Adding population data at 1km grid to finer grid

data_path = Path(data_inputs)

# Loading coarse pop data
pop_file = data_path / 'pereira_col_f_15_49_2015_1km.tif'
pop_raster, transform, crs = read_tif(pop_file)

# Converting the raster grid to vector data
pop_grid = raster2vector(pop_raster, transform, crs)
pop_grid = pop_grid.to_crs(epsg)
pop_grid['pop_grid_id'] = range(len(pop_grid))
# pop_grid.to_parquet(data_path / 'sanity_check_pop.parquet')

# Assign coarse population data to finer grid based on the centroid locations of the finer grid cells
grid['centroid'] = grid['geometry'].centroid
grid = gpd.sjoin(grid.set_geometry('centroid'), pop_grid, how='left', predicate='within')
print(grid.columns)
grid = grid[['grid_id', 'bcount', 'pop_grid_id', 'geometry','rowid', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max']]
grid.head()

Index(['grid_id', 'rowid', 'geometry', 'latitude', 'lat_min', 'lat_max',
       'longitude', 'lon_min', 'lon_max', 'bcount', 'centroid', 'index_right',
       'pop_grid_pop', 'pop_grid_id'],
      dtype='object')


,grid_id,bcount,pop_grid_id,geometry,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,0,0.0,24,"POLYGON ((428288.012 544939.864, 428281.792 54...",1,4.92732,4.926915,4.927725,-75.643963,-75.644491,-75.643434
1,1,0.0,24,"POLYGON ((428398.892 544939.757, 428392.673 54...",2,4.92732,4.926915,4.927725,-75.642963,-75.643491,-75.642434
2,2,0.0,25,"POLYGON ((428509.772 544939.65, 428503.553 545...",3,4.92732,4.926915,4.927725,-75.641963,-75.642491,-75.641434
3,3,0.0,25,"POLYGON ((428620.652 544939.543, 428614.433 54...",4,4.92732,4.926915,4.927725,-75.640962,-75.641491,-75.640434
4,4,0.0,25,"POLYGON ((428731.532 544939.437, 428725.313 54...",5,4.92732,4.926915,4.927725,-75.639962,-75.640491,-75.639434


In [22]:
# Calculate population weight (fraction of total population count that should be assigned to cell based on its building count)
grid_grouped_pop = grid.groupby('pop_grid_id')
building_count_pop = grid_grouped_pop['bcount'].sum().rename('pop_grid_bcount')
grid = grid.merge(building_count_pop, on='pop_grid_id', how='left')
grid['pop_weight'] = grid['bcount'] / grid['pop_grid_bcount']

# Compute disaggregated population count based on weight and building count at coarser cell level
grid = grid.merge(pop_grid, on='pop_grid_id', how='left')
grid['pop'] = grid['pop_grid_pop'] * grid['pop_weight']
grid.head()

,grid_id,bcount,pop_grid_id,geometry_x,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,geometry_y,pop
0,0,0.0,24,"POLYGON ((428288.012 544939.864, 428281.792 54...",1,4.92732,4.926915,4.927725,-75.643963,-75.644491,-75.643434,11.0,0.0,NaN,"POLYGON ((427514.248 545327.755, 428438.191 54...",NaN
1,1,0.0,24,"POLYGON ((428398.892 544939.757, 428392.673 54...",2,4.92732,4.926915,4.927725,-75.642963,-75.643491,-75.642434,11.0,0.0,NaN,"POLYGON ((427514.248 545327.755, 428438.191 54...",NaN
2,2,0.0,25,"POLYGON ((428509.772 544939.65, 428503.553 545...",3,4.92732,4.926915,4.927725,-75.641963,-75.642491,-75.641434,91.0,0.0,42.862034,"POLYGON ((428438.191 545326.859, 429362.132 54...",0.0
3,3,0.0,25,"POLYGON ((428620.652 544939.543, 428614.433 54...",4,4.92732,4.926915,4.927725,-75.640962,-75.641491,-75.640434,91.0,0.0,42.862034,"POLYGON ((428438.191 545326.859, 429362.132 54...",0.0
4,4,0.0,25,"POLYGON ((428731.532 544939.437, 428725.313 54...",5,4.92732,4.926915,4.927725,-75.639962,-75.640491,-75.639434,91.0,0.0,42.862034,"POLYGON ((428438.191 545326.859, 429362.132 54...",0.0


In [65]:
# Saving to file
grid = grid.drop(columns=["geometry_y"])
grid.head()


,grid_id,bcount,pop_grid_id,geometry_x,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop
0,0,0.0,24,"POLYGON ((428288.012 544939.864, 428281.792 54...",1,4.92732,4.926915,4.927725,-75.643963,-75.644491,-75.643434,11.0,0.0,NaN,NaN
1,1,0.0,24,"POLYGON ((428398.892 544939.757, 428392.673 54...",2,4.92732,4.926915,4.927725,-75.642963,-75.643491,-75.642434,11.0,0.0,NaN,NaN
2,2,0.0,25,"POLYGON ((428509.772 544939.65, 428503.553 545...",3,4.92732,4.926915,4.927725,-75.641963,-75.642491,-75.641434,91.0,0.0,42.862034,0.0
3,3,0.0,25,"POLYGON ((428620.652 544939.543, 428614.433 54...",4,4.92732,4.926915,4.927725,-75.640962,-75.641491,-75.640434,91.0,0.0,42.862034,0.0
4,4,0.0,25,"POLYGON ((428731.532 544939.437, 428725.313 54...",5,4.92732,4.926915,4.927725,-75.639962,-75.640491,-75.639434,91.0,0.0,42.862034,0.0


In [66]:
grid = grid.set_geometry("geometry_x")
grid = grid.to_crs(4326)
grid.to_file(data_temp + 'pop-grid-pereira-centroids.gpkg', driver='GPKG')

In [69]:
# if building data is of relevance. centroid gemetry to be deleted
buildings_footprint = buildings.drop(columns=['centroid'])
buildings_footprint.to_crs(4326)
buildings_footprint.to_file(data_temp + 'buildings-pereira.gpkg', driver='GPKG')

## 2. Spatial Analysis Pipeline

### Travel time and dista calculation using OpenRouteService (ORS)

Using OpenRouteService (ORS) Matrix API to calculate the travel time and distance from each population grid centroid to the healthcare facility. There are two options to process the time and distance calculations: Using the public ORS API or using a local instance of the ORS server.

note: this will generate a file 'OD_matrix_healthcare_pop_grid‘

In [ ]:
origin_gdf = population_centroids_gdf
origin_name_column = 'grid_code'
destination_gdf = healthcare_facilities_validated.dropna(subset=['geometry'])
destination_name_column = 'facility_name'

In [ ]:
origins = list(zip(origin_gdf.geometry.x, origin_gdf.geometry.y))

In [ ]:
destinations = list(zip(destination_gdf.geometry.x, destination_gdf.geometry.y))

In [ ]:
locations = origins + destinations

In [ ]:
origins_index = list(range(0, len(origins)))
destinations_index = list(range(len(origins), len(locations)))

In [ ]:
body = {'locations': locations,
       'destinations': destinations_index,
       'sources': origins_index,
       'metrics': ['distance', 'duration']}

headers = {
    'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
    'Authorization': api_key,
    'Content-Type': 'application/json; charset=utf-8'
}

response = requests.post('https://api.openrouteservice.org/v2/matrix/driving-car', json=body, headers=headers)

In [ ]:
distances = response.json().get('distances', [])
durations = response.json().get('durations', [])

In [ ]:
distances_duration_matrix = []

# Iterate over each origin (grid)
for origin_index, origin in origin_gdf.iterrows():
    origin_name = origin[origin_name_column]
    origin_x = origin.geometry.x
    origin_y = origin.geometry.y
    origin_distances = distances[origin_index]
    origin_durations = durations[origin_index]

    # find the minimum duration and the index of the minimum duration
    min_duration = min(origin_durations)
    min_index = origin_durations.index(min_duration)
    destination_index = destinations_index[min_index]
    dest_x, dest_y = locations[destination_index]
    filtered = healthcare_facilities_validated[(destination_gdf.geometry.x == dest_x) & (destination_gdf.geometry.y == dest_y) ]
    destination_row = filtered.iloc[0]
    dest_name = destination_row[destination_name_column]

        # Append both the distance and duration for this origin-destination pair
    distances_duration_matrix.append([
            origin_name, origin_y, origin_x,
            dest_name, dest_y, dest_x,
            min_duration
        ])

In [ ]:
# Convert the results into a DataFrame
matrix_df = pd.DataFrame(distances_duration_matrix, columns=[
    'grid_code','origin_lat', 'origin_lon',
    'destination_name', 'dest_lat', 'dest_lon','min_duration'
])

In [ ]:
# Save to CSV
merged_df = pd.merge(matrix_df, grid_df[['grid_code', 'population']], on='grid_code', how='left')
merged_df.to_csv(data_temp + 'distance_duration_matrix_temp.csv', index=False)

In [ ]:
merged_df

In [ ]:
geometry = [Point(xy) for xy in zip(merged_df['dest_lon'], merged_df['dest_lat'])]
gdf = gpd.GeoDataFrame(merged_df, geometry=geometry, crs="EPSG:4326")

gpkg_path = data_temp + 'distance_duration_matrix_temp.gpkg'
gdf.to_file(gpkg_path, layer="duration_matrix", driver="GPKG")

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
# Insert R code from the local ORS service
```

### Procedure for Computing the OD Matrix Using a Local Docker Environment

This section outlines the steps required to compute the Origin-Destination (OD) matrix using a local Docker environment. 

1. **Set Up Docker Environment**:

2. **Prepare Input Data**:

3. **Run the OD Matrix Computation Script**:

4. **Monitor the Process**:

5. **Retrieve and Validate Output**:

### Diego please add description here

## Processing OD Matrix

Population data is the result of combining 1km grid data with 100m grid data. See [Section 2]() for more details.

In [24]:
# If not loaded yet, read from the temporary folder
centroids_df = gpd.read_file(data_temp +'pop-grid-pereira-centroids.gpkg')
centroids_df

,grid_id,bcount,pop_grid_id,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max,pop_grid_bcount,pop_weight,pop_grid_pop,pop,geometry
0,0,0.0,24,1,4.927320,4.926915,4.927725,-75.643963,-75.644491,-75.643434,11.0,0.000000,NaN,NaN,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772..."
1,1,0.0,24,2,4.927320,4.926915,4.927725,-75.642963,-75.643491,-75.642434,11.0,0.000000,NaN,NaN,"POLYGON ((-75.64243 4.92691, -75.64249 4.92772..."
2,2,0.0,25,3,4.927320,4.926915,4.927725,-75.641963,-75.642491,-75.641434,91.0,0.000000,42.862034,0.000000,"POLYGON ((-75.64143 4.92691, -75.64149 4.92772..."
3,3,0.0,25,4,4.927320,4.926915,4.927725,-75.640962,-75.641491,-75.640434,91.0,0.000000,42.862034,0.000000,"POLYGON ((-75.64043 4.92691, -75.64049 4.92772..."
4,4,0.0,25,5,4.927320,4.926915,4.927725,-75.639962,-75.640491,-75.639434,91.0,0.000000,42.862034,0.000000,"POLYGON ((-75.63943 4.92691, -75.63949 4.92772..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25395,25395,3.0,762,25396,4.725662,4.725257,4.726067,-75.695086,-75.695613,-75.694559,51.0,0.058824,113.701828,6.688343,"POLYGON ((-75.69456 4.72526, -75.69461 4.72607..."
25396,25396,3.0,762,25397,4.725662,4.725257,4.726067,-75.694086,-75.694613,-75.693559,51.0,0.058824,113.701828,6.688343,"POLYGON ((-75.69356 4.72526, -75.69361 4.72607..."
25397,25397,1.0,762,25398,4.725662,4.725257,4.726067,-75.693086,-75.693613,-75.692559,51.0,0.019608,113.701828,2.229448,"POLYGON ((-75.69256 4.72526, -75.69261 4.72607..."
25398,25398,4.0,762,25399,4.725662,4.725257,4.726067,-75.692086,-75.692614,-75.691559,51.0,0.078431,113.701828,8.917790,"POLYGON ((-75.69156 4.72526, -75.69161 4.72607..."


In [26]:
# If not loaded yet, read from the temporary folder
matrix_df = pd.read_csv(data_temp +'pereira_access.csv')
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,1,0,1798.22,22.08
1,1,1,1799.13,22.09
2,1,2,1800.90,22.11
3,1,3,1804.26,22.13
4,1,4,1721.53,21.39
...,...,...,...,...
330195,13,25395,1960.07,21.16
330196,13,25396,1967.19,21.26
330197,13,25397,1982.61,21.47
330198,13,25398,1989.63,21.56


**GRID CELLS WITHOUT TRAVEL TIME ESTIMATE**

If a grid cell has a NULL value in the travel estimate, we will remove it from the analysis. This is because we cannot calculate the 2SFCA without a travel time estimate.

In [27]:
# Removing rows with NaN values in the 'duration_seconds' column
matrix_df = matrix_df.dropna(subset=['duration_seconds'])
matrix_df

,origin_id,destination_id,duration_seconds,distance_km
0,1,0,1798.22,22.08
1,1,1,1799.13,22.09
2,1,2,1800.90,22.11
3,1,3,1804.26,22.13
4,1,4,1721.53,21.39
...,...,...,...,...
330195,13,25395,1960.07,21.16
330196,13,25396,1967.19,21.26
330197,13,25397,1982.61,21.47
330198,13,25398,1989.63,21.56


To process the OD Matrix we need merge it to create an integrated dataset that combines data from the healthcare facilities and population grid.For doing so, we will use the pandas library and join functions based on the id columns of all datasets.

In [28]:
pop_centroids_hcf = pd.merge(matrix_df, centroids_df[['rowid', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','bcount','pop_grid_bcount', 'pop_grid_pop', 'pop', 'geometry']], 
                     left_on='destination_id', right_on='rowid', how='left')
pop_centroids_hcf

,origin_id,destination_id,duration_seconds,distance_km,rowid,longitude,latitude,lon_min,lat_min,lon_max,lat_max,bcount,pop_grid_bcount,pop_grid_pop,pop,geometry
0,1,0,1798.22,22.08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,1,1,1799.13,22.09,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,0.0,11.0,NaN,NaN,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772..."
2,1,2,1800.90,22.11,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,0.0,11.0,NaN,NaN,"POLYGON ((-75.64243 4.92691, -75.64249 4.92772..."
3,1,3,1804.26,22.13,3.0,-75.641963,4.927320,-75.642491,4.926915,-75.641434,4.927725,0.0,91.0,42.862034,0.000000,"POLYGON ((-75.64143 4.92691, -75.64149 4.92772..."
4,1,4,1721.53,21.39,4.0,-75.640962,4.927320,-75.641491,4.926915,-75.640434,4.927725,0.0,91.0,42.862034,0.000000,"POLYGON ((-75.64043 4.92691, -75.64049 4.92772..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330195,13,25395,1960.07,21.16,25395.0,-75.696086,4.725662,-75.696613,4.725257,-75.695559,4.726067,4.0,51.0,113.701828,8.917790,"POLYGON ((-75.69556 4.72526, -75.69561 4.72607..."
330196,13,25396,1967.19,21.26,25396.0,-75.695086,4.725662,-75.695613,4.725257,-75.694559,4.726067,3.0,51.0,113.701828,6.688343,"POLYGON ((-75.69456 4.72526, -75.69461 4.72607..."
330197,13,25397,1982.61,21.47,25397.0,-75.694086,4.725662,-75.694613,4.725257,-75.693559,4.726067,3.0,51.0,113.701828,6.688343,"POLYGON ((-75.69356 4.72526, -75.69361 4.72607..."
330198,13,25398,1989.63,21.56,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,1.0,51.0,113.701828,2.229448,"POLYGON ((-75.69256 4.72526, -75.69261 4.72607..."


In [29]:
pop_centroids_hcf = pop_centroids_hcf.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    "rowid": "grid_id",
    "origin_id": "hcf_uid",
    "pop": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min","origin_lat_min","origin_lon_max","origin_lat_max","population", "bcount","pop_grid_bcount", "pop_grid_pop","geometry", "hcf_uid", "duration_seconds", "distance_km"]
pop_centroids_hcf = pop_centroids_hcf[columns_to_keep]

In [30]:
pop_centroids_hcf

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,hcf_uid,duration_seconds,distance_km
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,1,1798.22,22.08
1,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,NaN,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772...",1,1799.13,22.09
2,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,NaN,"POLYGON ((-75.64243 4.92691, -75.64249 4.92772...",1,1800.90,22.11
3,3.0,-75.641963,4.927320,-75.642491,4.926915,-75.641434,4.927725,0.000000,0.0,91.0,42.862034,"POLYGON ((-75.64143 4.92691, -75.64149 4.92772...",1,1804.26,22.13
4,4.0,-75.640962,4.927320,-75.641491,4.926915,-75.640434,4.927725,0.000000,0.0,91.0,42.862034,"POLYGON ((-75.64043 4.92691, -75.64049 4.92772...",1,1721.53,21.39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330195,25395.0,-75.696086,4.725662,-75.696613,4.725257,-75.695559,4.726067,8.917790,4.0,51.0,113.701828,"POLYGON ((-75.69556 4.72526, -75.69561 4.72607...",13,1960.07,21.16
330196,25396.0,-75.695086,4.725662,-75.695613,4.725257,-75.694559,4.726067,6.688343,3.0,51.0,113.701828,"POLYGON ((-75.69456 4.72526, -75.69461 4.72607...",13,1967.19,21.26
330197,25397.0,-75.694086,4.725662,-75.694613,4.725257,-75.693559,4.726067,6.688343,3.0,51.0,113.701828,"POLYGON ((-75.69356 4.72526, -75.69361 4.72607...",13,1982.61,21.47
330198,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,113.701828,"POLYGON ((-75.69256 4.72526, -75.69261 4.72607...",13,1989.63,21.56


Merging the dataframe than contains the od matrix (with the healthcare facility class) and the population data with the full information about health care facilities.

In [35]:
# For Pereira, the HCF_ID was generated at the time of processing the OD Matrix. 
# Therefore, we need to read the file that contains those ids
healthcare_facilities_validated = gpd.read_file(data_temp + 'healthcare_facilities_emoc_hcfid.geojson')

healthcare_facilities_validated = healthcare_facilities_validated.rename(columns={
    "NombreSede": "facility_name",
    "project_validation": "Local_Validation"
})

In [36]:
distances_duration_matrix = pd.merge(pop_centroids_hcf, healthcare_facilities_validated[['hcf_id','facility_name', 'longitude', 'latitude', 'Local_Validation']], 
                     left_on='hcf_uid', right_on='hcf_id', how='left')

In [37]:
distances_duration_matrix = distances_duration_matrix.rename(columns={
    "longitude": "dest_lon",
    "latitude": "dest_lat"
})
distances_duration_matrix = distances_duration_matrix.drop(columns=['hcf_uid'])

In [38]:
category_counts = healthcare_facilities_validated['Local_Validation'].value_counts()
print(category_counts)

Local_Validation
Private Basic EmOC            8
Public Comprehensive EmOC     3
Private Comprehensive EmOC    2
Name: count, dtype: int64


In [39]:
distances_duration_matrix['Local_Validation'] = distances_duration_matrix['Local_Validation'].replace({
    'Public/Private Basic EmOC': 'Private Basic EmOC',
    'Public/Private comprehensive EmOC (missionary Hospital)': 'Private Comprehensive EmOC'
})

In [40]:
selected_categories = ['Public Comprehensive EmOC', 'Private Comprehensive EmOC', 
                       'Private Basic EmOC', 'Public Basic EmOC']

In [41]:
distances_duration_matrix = distances_duration_matrix[
    distances_duration_matrix['Local_Validation'].isin(selected_categories)]

distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,1798.22,22.08,1,CLINICA COMFAMILIAR,-75.680904,4.807561,Private Comprehensive EmOC
1,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,NaN,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772...",1799.13,22.09,1,CLINICA COMFAMILIAR,-75.680904,4.807561,Private Comprehensive EmOC
2,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,NaN,"POLYGON ((-75.64243 4.92691, -75.64249 4.92772...",1800.90,22.11,1,CLINICA COMFAMILIAR,-75.680904,4.807561,Private Comprehensive EmOC
3,3.0,-75.641963,4.927320,-75.642491,4.926915,-75.641434,4.927725,0.000000,0.0,91.0,42.862034,"POLYGON ((-75.64143 4.92691, -75.64149 4.92772...",1804.26,22.13,1,CLINICA COMFAMILIAR,-75.680904,4.807561,Private Comprehensive EmOC
4,4.0,-75.640962,4.927320,-75.641491,4.926915,-75.640434,4.927725,0.000000,0.0,91.0,42.862034,"POLYGON ((-75.64043 4.92691, -75.64049 4.92772...",1721.53,21.39,1,CLINICA COMFAMILIAR,-75.680904,4.807561,Private Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
330195,25395.0,-75.696086,4.725662,-75.696613,4.725257,-75.695559,4.726067,8.917790,4.0,51.0,113.701828,"POLYGON ((-75.69556 4.72526, -75.69561 4.72607...",1960.07,21.16,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC
330196,25396.0,-75.695086,4.725662,-75.695613,4.725257,-75.694559,4.726067,6.688343,3.0,51.0,113.701828,"POLYGON ((-75.69456 4.72526, -75.69461 4.72607...",1967.19,21.26,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC
330197,25397.0,-75.694086,4.725662,-75.694613,4.725257,-75.693559,4.726067,6.688343,3.0,51.0,113.701828,"POLYGON ((-75.69356 4.72526, -75.69361 4.72607...",1982.61,21.47,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC
330198,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,113.701828,"POLYGON ((-75.69256 4.72526, -75.69261 4.72607...",1989.63,21.56,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC


In [44]:
# creat subsets based on categories of 'Validation of HCFs Categorization'
categories = {
    "public_comprehensive_EmOC": ["Public Comprehensive EmOC"],
    "private_comprehensive_EmOC": ["Private Comprehensive EmOC"],
    "private_basic_EmOC": ["Private Basic EmOC"],
    "public_basic_EmOC": ["Public Basic EmOC"]
}

subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['Local_Validation'].str.contains('|'.join(values), na=False)
    ]
    for key, values in categories.items()
}

public_CEmOC = subsets["public_comprehensive_EmOC"]
private_CEmOC = subsets["private_comprehensive_EmOC"]
public_BEmOC = subsets["public_basic_EmOC"]
private_BEmOC = subsets["private_basic_EmOC"]

In [45]:
# Step 2: Define a function to get 3 smallest duration_seconds per grid_id for each category
def get_closest_3(df, n=3):
    return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)

In [46]:
# Step 3: If the subsets are already created for each category, we apply the function to each subset:
public_CEmOC_closest_3 = get_closest_3(public_CEmOC)
private_CEmOC_closest_3 = get_closest_3(private_CEmOC)
public_BEmOC_closest_3 = get_closest_3(public_BEmOC)
private_BEmOC_closest_3 = get_closest_3(private_BEmOC)

/var/folders/0s/jzqshn192vjfnj70wgny2bq00000gn/T/ipykernel_5104/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_seconds')).reset_index(drop=True)
/var/folders/0s/jzqshn192vjfnj70wgny2bq00000gn/T/ipykernel_5104/1217296076.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmal

In [47]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    public_CEmOC_closest_3, private_CEmOC_closest_3,
    public_BEmOC_closest_3, private_BEmOC_closest_3
])
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,pop_grid_pop,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation
0,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,NaN,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772...",1984.59,23.21,6,EMPRESA SOCIAL DEL ESTADO HOSPITAL UNIVERSITAR...,-75.698894,4.818055,Public Comprehensive EmOC
1,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,NaN,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772...",2178.15,27.76,3,HOSPITAL DE KENNEDY,-75.733243,4.815743,Public Comprehensive EmOC
2,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,NaN,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772...",2474.26,28.52,4,HOSPITAL SAN JOAQUIN,-75.742104,4.797746,Public Comprehensive EmOC
3,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,NaN,"POLYGON ((-75.64243 4.92691, -75.64249 4.92772...",1986.36,23.23,6,EMPRESA SOCIAL DEL ESTADO HOSPITAL UNIVERSITAR...,-75.698894,4.818055,Public Comprehensive EmOC
4,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,NaN,"POLYGON ((-75.64243 4.92691, -75.64249 4.92772...",2179.92,27.78,3,HOSPITAL DE KENNEDY,-75.733243,4.815743,Public Comprehensive EmOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76192,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,113.701828,"POLYGON ((-75.69256 4.72526, -75.69261 4.72607...",1989.63,21.56,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC
76193,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,113.701828,"POLYGON ((-75.69256 4.72526, -75.69261 4.72607...",2041.62,26.12,12,CLINICA CRECER IPS SAS,-75.725151,4.822905,Private Basic EmOC
76194,25399.0,-75.692086,4.725662,-75.692614,4.725257,-75.691559,4.726067,8.917790,4.0,51.0,113.701828,"POLYGON ((-75.69156 4.72526, -75.69161 4.72607...",1552.42,18.27,9,Clinica San Rafael sede CUBA,-75.741407,4.801610,Private Basic EmOC
76195,25399.0,-75.692086,4.725662,-75.692614,4.725257,-75.691559,4.726067,8.917790,4.0,51.0,113.701828,"POLYGON ((-75.69156 4.72526, -75.69161 4.72607...",2001.74,21.74,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC


In [48]:
geometry = [Point(xy) for xy in zip(distances_duration_matrix['origin_lon'], distances_duration_matrix['origin_lat'])]
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry=geometry, crs="EPSG:4326")

In [49]:
gpkg_path = data_temp + 'distances_duration_3_closet_Emoc.gpkg'
gdf.to_file(gpkg_path, layer="distances_duration_3_closet_Emoc", driver="GPKG")

In [50]:
# Review and remove
origin_dest = distances_duration_matrix

## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [51]:
# Function
from math import *
d = 10 * 60 # try max duration 5/10mins/15mins/20 car, under estimation of travel time and traffic condition realted to the selected data sourse 
W = 0.01
beta = - d ** 2 / log(W)
print(beta)

78173.00674258533


In [52]:
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0      1.0  -75.643963     4.92732      -75.644491        4.926915   
1      1.0  -75.643963     4.92732      -75.644491        4.926915   
2      1.0  -75.643963     4.92732      -75.644491        4.926915   
3      2.0  -75.642963     4.92732      -75.643491        4.926915   
4      2.0  -75.642963     4.92732      -75.643491        4.926915   

   origin_lon_max  origin_lat_max  population  bcount  pop_grid_bcount  \
0      -75.643434        4.927725         NaN     0.0             11.0   
1      -75.643434        4.927725         NaN     0.0             11.0   
2      -75.643434        4.927725         NaN     0.0             11.0   
3      -75.642434        4.927725         NaN     0.0             11.0   
4      -75.642434        4.927725         NaN     0.0             11.0   

   pop_grid_pop                                           geometry  \
0           NaN  POLYGON ((-75.64343 4.92691, -75.64349 4.92772.

In [53]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [54]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')
origin_dest_acc = origin_dest  # Backup

In [55]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact.
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [56]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [57]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,geometry,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,Weight,Pop_W
0,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,...,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772...",1984.59,23.21,6,EMPRESA SOCIAL DEL ESTADO HOSPITAL UNIVERSITAR...,-75.698894,4.818055,Public Comprehensive EmOC,0.0,NaN
1,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,...,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772...",2178.15,27.76,3,HOSPITAL DE KENNEDY,-75.733243,4.815743,Public Comprehensive EmOC,0.0,NaN
2,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,...,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772...",2474.26,28.52,4,HOSPITAL SAN JOAQUIN,-75.742104,4.797746,Public Comprehensive EmOC,0.0,NaN
3,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,...,"POLYGON ((-75.64243 4.92691, -75.64249 4.92772...",1986.36,23.23,6,EMPRESA SOCIAL DEL ESTADO HOSPITAL UNIVERSITAR...,-75.698894,4.818055,Public Comprehensive EmOC,0.0,NaN
4,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,...,"POLYGON ((-75.64243 4.92691, -75.64249 4.92772...",2179.92,27.78,3,HOSPITAL DE KENNEDY,-75.733243,4.815743,Public Comprehensive EmOC,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76192,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,...,"POLYGON ((-75.69256 4.72526, -75.69261 4.72607...",1989.63,21.56,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC,0.0,0.0
76193,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,...,"POLYGON ((-75.69256 4.72526, -75.69261 4.72607...",2041.62,26.12,12,CLINICA CRECER IPS SAS,-75.725151,4.822905,Private Basic EmOC,0.0,0.0
76194,25399.0,-75.692086,4.725662,-75.692614,4.725257,-75.691559,4.726067,8.917790,4.0,51.0,...,"POLYGON ((-75.69156 4.72526, -75.69161 4.72607...",1552.42,18.27,9,Clinica San Rafael sede CUBA,-75.741407,4.801610,Private Basic EmOC,0.0,0.0
76195,25399.0,-75.692086,4.725662,-75.692614,4.725257,-75.691559,4.726067,8.917790,4.0,51.0,...,"POLYGON ((-75.69156 4.72526, -75.69161 4.72607...",2001.74,21.74,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC,0.0,0.0


In [58]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='hcf_id')['Pop_W'].sum().reset_index()

In [59]:
origin_dest_sum

,hcf_id,Pop_W
0,1,12084.536666
1,2,13788.389923
2,3,4676.136752
3,4,10986.402819
4,5,8944.685965
5,6,10481.907044
6,7,10117.294949
7,8,7058.312935
8,9,7175.437491
9,10,10849.090092


In [60]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='hcf_id')

In [61]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,duration_seconds,distance_km,hcf_id,facility_name,dest_lon,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_y
0,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,...,1984.59,23.21,6,EMPRESA SOCIAL DEL ESTADO HOSPITAL UNIVERSITAR...,-75.698894,4.818055,Public Comprehensive EmOC,0.0,NaN,10481.907044
1,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,...,2178.15,27.76,3,HOSPITAL DE KENNEDY,-75.733243,4.815743,Public Comprehensive EmOC,0.0,NaN,4676.136752
2,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,...,2474.26,28.52,4,HOSPITAL SAN JOAQUIN,-75.742104,4.797746,Public Comprehensive EmOC,0.0,NaN,10986.402819
3,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,...,1986.36,23.23,6,EMPRESA SOCIAL DEL ESTADO HOSPITAL UNIVERSITAR...,-75.698894,4.818055,Public Comprehensive EmOC,0.0,NaN,10481.907044
4,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,...,2179.92,27.78,3,HOSPITAL DE KENNEDY,-75.733243,4.815743,Public Comprehensive EmOC,0.0,NaN,4676.136752
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203187,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,...,1989.63,21.56,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC,0.0,0.0,7634.550868
203188,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,...,2041.62,26.12,12,CLINICA CRECER IPS SAS,-75.725151,4.822905,Private Basic EmOC,0.0,0.0,4572.930862
203189,25399.0,-75.692086,4.725662,-75.692614,4.725257,-75.691559,4.726067,8.917790,4.0,51.0,...,1552.42,18.27,9,Clinica San Rafael sede CUBA,-75.741407,4.801610,Private Basic EmOC,0.0,0.0,7175.437491
203190,25399.0,-75.692086,4.725662,-75.692614,4.725257,-75.691559,4.726067,8.917790,4.0,51.0,...,2001.74,21.74,13,SERVICIOS DE EMERGENCIAS MEDICAS GUADALUPE SAS,-75.709662,4.817263,Private Basic EmOC,0.0,0.0,7634.550868


In [62]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [63]:
supply_map = {
    'Public Comprehensive EmOC': 1,
    'Private Comprehensive EmOC': 0.7,
    'Public Basic EmOC': 0.5,
    'Private Basic EmOC': 0.35
}

In [64]:
origin_dest_acc['supply'] = origin_dest_acc['Local_Validation'].map(supply_map)
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)

/var/folders/0s/jzqshn192vjfnj70wgny2bq00000gn/T/ipykernel_5104/2370967217.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)


In [65]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [66]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [67]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])

In [68]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,bcount,pop_grid_bcount,...,dest_lat,Local_Validation,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,...,4.818055,Public Comprehensive EmOC,0.0,NaN,10481.907044,1.00,0.000095,0.0,0.0,0.0
1,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,...,4.815743,Public Comprehensive EmOC,0.0,NaN,4676.136752,1.00,0.000214,0.0,0.0,0.0
2,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,NaN,0.0,11.0,...,4.797746,Public Comprehensive EmOC,0.0,NaN,10986.402819,1.00,0.000091,0.0,0.0,0.0
3,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,...,4.818055,Public Comprehensive EmOC,0.0,NaN,10481.907044,1.00,0.000095,0.0,0.0,0.0
4,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,NaN,0.0,11.0,...,4.815743,Public Comprehensive EmOC,0.0,NaN,4676.136752,1.00,0.000214,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203187,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,...,4.817263,Private Basic EmOC,0.0,0.0,7634.550868,0.35,0.000046,0.0,0.0,0.0
203188,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,2.229448,1.0,51.0,...,4.822905,Private Basic EmOC,0.0,0.0,4572.930862,0.35,0.000077,0.0,0.0,0.0
203189,25399.0,-75.692086,4.725662,-75.692614,4.725257,-75.691559,4.726067,8.917790,4.0,51.0,...,4.801610,Private Basic EmOC,0.0,0.0,7175.437491,0.35,0.000049,0.0,0.0,0.0
203190,25399.0,-75.692086,4.725662,-75.692614,4.725257,-75.691559,4.726067,8.917790,4.0,51.0,...,4.817263,Private Basic EmOC,0.0,0.0,7634.550868,0.35,0.000046,0.0,0.0,0.0


In [70]:
max(origin_dest_acc.Accessibility_standard)

1.0

In [71]:
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_3closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_3closest", driver="GPKG")

# 4. Grouping by grid ID to prepare the final output file
There is a need to update this part of the code

In [73]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_3closest.gpkg')

In [74]:
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry']]

In [75]:
# Group by multiple columns and calculate the mean for numeric columns
# results_grid = results_grid.groupby(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard']).count().reset_index()
results_grid = results_grid.drop_duplicates(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry'])
type(results_grid)

geopandas.geodataframe.GeoDataFrame

In [76]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access', driver='GPKG')

In [77]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry
0,1.0,-75.643963,4.927320,-75.644491,4.926915,-75.643434,4.927725,0.0,"POLYGON ((-75.64343 4.92691, -75.64349 4.92772..."
3,2.0,-75.642963,4.927320,-75.643491,4.926915,-75.642434,4.927725,0.0,"POLYGON ((-75.64243 4.92691, -75.64249 4.92772..."
6,3.0,-75.641963,4.927320,-75.642491,4.926915,-75.641434,4.927725,0.0,"POLYGON ((-75.64143 4.92691, -75.64149 4.92772..."
9,4.0,-75.640962,4.927320,-75.641491,4.926915,-75.640434,4.927725,0.0,"POLYGON ((-75.64043 4.92691, -75.64049 4.92772..."
12,5.0,-75.639962,4.927320,-75.640491,4.926915,-75.639434,4.927725,0.0,"POLYGON ((-75.63943 4.92691, -75.63949 4.92772..."
...,...,...,...,...,...,...,...,...,...
76182,25395.0,-75.696086,4.725662,-75.696613,4.725257,-75.695559,4.726067,0.0,"POLYGON ((-75.69556 4.72526, -75.69561 4.72607..."
76185,25396.0,-75.695086,4.725662,-75.695613,4.725257,-75.694559,4.726067,0.0,"POLYGON ((-75.69456 4.72526, -75.69461 4.72607..."
76188,25397.0,-75.694086,4.725662,-75.694613,4.725257,-75.693559,4.726067,0.0,"POLYGON ((-75.69356 4.72526, -75.69361 4.72607..."
76191,25398.0,-75.693086,4.725662,-75.693613,4.725257,-75.692559,4.726067,0.0,"POLYGON ((-75.69256 4.72526, -75.69261 4.72607..."


### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  

In [78]:
results_grid['result'] = -1
results_grid.loc[results_grid['Accessibility_standard'] > 0.000001, 'result'] = 2
results_grid.loc[results_grid['Accessibility_standard'] > 0.005, 'result'] = 1
results_grid.loc[results_grid['Accessibility_standard'] > 0.02, 'result'] = 0

In [79]:
category_counts = results_grid['result'].value_counts()
print(category_counts)

result
 2    9465
-1    7478
 0    6265
 1    2191
Name: count, dtype: int64


### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [80]:
results_grid['focused'] = 0
# Focus areas between the Low category and the excluded cells due to low population or no buildings
results_grid.loc[(results_grid['Accessibility_standard'] > 0.000001) & (results_grid['Accessibility_standard'] < 0.0000015), 'focused'] = 1
# Focus areas between the Medium and High categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.003) & (results_grid['Accessibility_standard'] < 0.006), 'focused'] = 1
# Focus areas between the Low and Medium categories
results_grid.loc[(results_grid['Accessibility_standard'] > 0.019) & (results_grid['Accessibility_standard'] < 0.03), 'focused'] = 1

In [81]:
category_counts = results_grid['focused'].value_counts()
print(category_counts)

focused
0    23126
1     2273
Name: count, dtype: int64


In [82]:
results_grid = results_grid.loc[results_grid['result'] != -1]

In [83]:
results_grid = results_grid.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

In [84]:
results_grid

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,Accessibility_standard,geometry,result,focused
9021,3008.0,-75.673005,4.870627,-75.673533,4.870222,-75.672477,4.871031,0.000002,"POLYGON ((-75.67248 4.87022, -75.67253 4.87103...",2,0
9024,3009.0,-75.672005,4.870627,-75.672533,4.870222,-75.671477,4.871031,0.000002,"POLYGON ((-75.67148 4.87022, -75.67153 4.87103...",2,0
9258,3087.0,-75.673949,4.869817,-75.674477,4.869412,-75.673420,4.870222,0.000002,"POLYGON ((-75.67342 4.86941, -75.67348 4.87022...",2,0
9261,3088.0,-75.672949,4.869817,-75.673477,4.869412,-75.672420,4.870222,0.000002,"POLYGON ((-75.67242 4.86941, -75.67248 4.87022...",2,0
9264,3089.0,-75.671949,4.869817,-75.672477,4.869412,-75.671420,4.870222,0.000002,"POLYGON ((-75.67142 4.86941, -75.67148 4.87022...",2,0
...,...,...,...,...,...,...,...,...,...,...,...
75771,25258.0,-75.713248,4.728092,-75.713775,4.727687,-75.712720,4.728497,0.000001,"POLYGON ((-75.71272 4.72769, -75.71277 4.7285,...",2,1
75774,25259.0,-75.712248,4.728092,-75.712775,4.727687,-75.711720,4.728497,0.000001,"POLYGON ((-75.71172 4.72769, -75.71178 4.7285,...",2,1
75777,25260.0,-75.711248,4.728092,-75.711775,4.727687,-75.710721,4.728497,0.000001,"POLYGON ((-75.71072 4.72769, -75.71078 4.7285,...",2,1
75873,25292.0,-75.719192,4.727282,-75.719719,4.726877,-75.718665,4.727687,0.000004,"POLYGON ((-75.71866 4.72688, -75.71872 4.72769...",2,0


In [85]:
# Save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'emergency-maternal-care-deprivation-access-class.gpkg'
results_grid.to_file(output_gpkg_path, layer='emergency-maternal-care-deprivation-access-class', driver='GPKG')

In [86]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_table = results_grid.drop(columns=['Accessibility_standard', 'grid_id', 'geometry'])
results_table.to_csv(model_outputs + 'model-output.csv', index=False)